In [4]:
import sys
from pathlib import Path
import torch
import torch.nn as nn


project_root = Path("/mnt/filesystem-test/nmr")
sys.path.insert(0, str(project_root / "scripts"))

from model import MAX_J_VALUES, MULTIPLICITY_TO_ID
from model.embedding import NmrEmbedder, NmrBatch
from model import FoundationNMRProcessor, PairedFoundationDataset
from torch.utils.data import DataLoader
from data import CanonicalParquetDataset, CANONICAL_MULTIPLICITIES
from model import FoundationNMRProcessor, H_AVAILABILITY_FIELDS, MAX_J_VALUES, MULTIPLICITY_TO_ID
from scripts.model.utils.fourier_features import FourierFeatures, RBFExpansion

dataset_path = project_root / "datasets/cleaned/test_benchmark.parquet"
assert dataset_path.is_file(), f"Dataset non trovato: {dataset_path}"

canonical_dataset = CanonicalParquetDataset(dataset_path)
processor = FoundationNMRProcessor()

canonical_loader = DataLoader(
    canonical_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0,
    collate_fn=processor,
)

canonical_batch = next(iter(canonical_loader))


dataset = PairedFoundationDataset(
    nmr_path="datasets/cleaned/train_val.parquet",
    molecular_properties_path=(
        "datasets/cleaned/train_val_mol_properties.parquet"
    ),
    arrow_batch_size=2048,
)

loader = DataLoader(
    dataset,
    batch_size=8,
    num_workers=4,
    collate_fn=FoundationNMRProcessor(),
    pin_memory=True,
    persistent_workers=True,
)

batch = next(iter(loader))

print("Record IDs:")
print(batch["record_ids"])

print("\nTensor shapes:")
for modality in ("h", "c"):
    for name, tensor in batch[modality].items():
        print(f"{modality}.{name:18s} {tuple(tensor.shape)}  {tensor.dtype}")

print("\nNumero di picchi reali per record:")
print("H:", batch["h"]["peak_mask"].sum(dim=1).tolist())
print("C:", batch["c"]["peak_mask"].sum(dim=1).tolist())

print("\nAvailability H:")
for channel, name in enumerate(H_AVAILABILITY_FIELDS):
    available = batch["h"]["availability"][:, :, channel].sum(dim=1)
    print(f"{name:18s}: {available.tolist()}")

# Ispeziona i valori reali del primo record.
h_mask = batch["h"]["peak_mask"][0]
c_mask = batch["c"]["peak_mask"][0]

print("\nPrimo record:", batch["record_ids"][0])
print("H shifts:", batch["h"]["shift"][0][h_mask].tolist())
print("C shifts:", batch["c"]["shift"][0][c_mask].tolist())
print("H integration:", batch["h"]["integration"][0][h_mask].tolist())
print("H multiplicity IDs:", batch["h"]["multiplicity"][0][h_mask].tolist())
print("H range half-span:", batch["h"]["range_half_span"][0][h_mask].tolist())

assert batch["h"]["shift"].shape == (8, 60)
assert batch["h"]["j_values"].shape == (8, 60, 6)
assert batch["h"]["availability"].shape == (8, 60, 4)
assert batch["c"]["shift"].shape == (8, 60)

print("\n✓ Primo batch reale processato correttamente")

Record IDs:
['nmrpeak-mst-nmr:train:0', 'nmrpeak-mst-nmr:train:1', 'nmrpeak-mst-nmr:train:10', 'nmrpeak-mst-nmr:train:100', 'nmrpeak-mst-nmr:train:1000', 'nmrpeak-mst-nmr:train:10000', 'nmrpeak-mst-nmr:train:100000', 'nmrpeak-mst-nmr:train:100001']

Tensor shapes:
h.shift              (8, 60)  torch.float32
h.integration        (8, 60)  torch.float32
h.multiplicity       (8, 60)  torch.int64
h.j_values           (8, 60, 6)  torch.float32
h.range_half_span    (8, 60)  torch.float32
h.peak_mask          (8, 60)  torch.bool
h.j_mask             (8, 60, 6)  torch.bool
h.availability       (8, 60, 4)  torch.bool
c.shift              (8, 60)  torch.float32
c.peak_mask          (8, 60)  torch.bool

Numero di picchi reali per record:
H: [9, 6, 7, 12, 7, 7, 8, 8]
C: [13, 11, 9, 20, 11, 11, 18, 18]

Availability H:
integration       : [9, 6, 7, 12, 7, 7, 8, 8]
multiplicity      : [9, 6, 7, 12, 7, 7, 8, 8]
j_values          : [9, 6, 7, 12, 7, 7, 8, 8]
range_half_span   : [9, 6, 7, 12, 7, 7, 8, 8]

In [10]:
batch['fingerprints'].shape

torch.Size([8, 2048])

In [6]:
from pathlib import Path
from dataclasses import asdict
import yaml
from scripts.model.FoMoNMR import FoMoNMR

model = FoMoNMR.from_config_file(
    "scripts/model/configs/default.yaml"
)

/home/nicola-greco/.venvs/nmr/nmr_venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
pooled, nmr_peaks, nmr_peak_valid_mask = model(batch)

In [25]:
pooled.shape, nmr_peaks.shape, nmr_peak_valid_mask.shape

(torch.Size([8, 512]), torch.Size([8, 120, 512]), torch.Size([8, 120]))